In [1]:
import random
import pandas as pd

random.seed(11)

machines = pd.DataFrame({
    "machine_id": ["M01", "M02", "M03", "M04", "M05", "M06"],
    "type": ["CNC", "CNC", "Press", "Press", "Welder", "Welder"],
    "plant": ["North", "North", "North", "South", "South", "South"],
    "install_year": [2015, 2019, 2012, 2020, 2017, 2022],
})
machines.to_csv("machines.csv", index=False)

base_rate = {"CNC": 120, "Press": 200, "Welder": 80}

rows = []
for d in pd.date_range("2025-01-01", "2025-12-31"):
    for _, m in machines.iterrows():
        if random.random() < 0.05:
            continue
        shift = random.choice(["Day", "Night"])
        age = 2025 - m["install_year"]
        rate = base_rate[m["type"]] * (1 - 0.01 * age) * (0.92 if shift == "Night" else 1.0)
        units = max(1, round(random.gauss(rate, rate * 0.1)))
        defect_p = 0.02 + 0.004 * age + (0.01 if shift == "Night" else 0)
        defects = min(units, max(0, round(random.gauss(units * defect_p, 1.5))))
        downtime = max(0, round(random.gauss(30 + 5 * age, 15)))
        rows.append([d.strftime("%Y-%m-%d"), m["machine_id"], shift, units, defects, downtime])

log = pd.DataFrame(rows, columns=["date", "machine_id", "shift", "units", "defects", "downtime_min"])

for i in random.sample(range(len(log)), 25):
    log.loc[i, "date"] = pd.to_datetime(log.loc[i, "date"]).strftime("%d.%m.%Y")
for i in random.sample(range(len(log)), 60):
    log.loc[i, "shift"] = random.choice([log.loc[i, "shift"].lower(), log.loc[i, "shift"].upper()])
for i in random.sample(range(len(log)), 20):
    log.loc[i, "units"] = None
for i in random.sample(range(len(log)), 8):
    log.loc[i, "units"] = -1
for i in random.sample(range(len(log)), 15):
    log.loc[i, "machine_id"] = "M07"

log = pd.concat([log, log.sample(12, random_state=5)]).reset_index(drop=True)
log.to_csv("production_log.csv", index=False)